Leticia Barbanera, 14588642

João Rissi, 14582823

Este notebook foi desenvolvido para automatizar a classificação de petições iniciais em múltiplas classes do direito. Em vez de utilizar abordagens tradicionais de classificação que sofrem com textos longos e vocabulário complexo, adotei uma arquitetura baseada em **Dual Encoders** (Bi-Encoders) combinada com busca vetorial de alta performance (com FAISS).

A estratégia consiste em realizar um *fine-tuning* no modelo de linguagem **BERTimbau** utilizando aprendizado contrastivo com a função de perda *Multiple Negatives Ranking Loss (MNRL)*. O modelo aprende a aproximar os vetores de representação das petições iniciais dos vetores das descrições enriquecidas de cada classe jurídica. Após o treinamento, essas classes são mapeadas em um espaço vetorial indexado pelo **FAISS**, permitindo consultas em tempo real por similaridade de cosseno. Por fim, realizei uma varredura na base de validação para definir o melhor *threshold*, já que dado o desbalanceamento das classes, provavelmente nem teriam o mesmo limiar.

In [ ]:
#!pip install pandas sentence-transformers datasets tqdm faiss-cpu

In [ ]:
import torch
import gc
#esvazia o cache interno de memória da NVIDIAGPU
gc.collect()
torch.cuda.empty_cache()

🧹 Memória RAM e GPU limpas com sucesso!


In [ ]:
import os
import re
import time
import gc
import pandas as pd
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from google.colab import files

#usei gpu no colab
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device.upper()}")

In [ ]:
#limpar e padronizar as classes do dataset
def limpar_labels_zoados(texto_label):
    if isinstance(texto_label, list):
        return [str(l).strip().upper() for l in texto_label]

    texto_str = str(texto_label)
    texto_str = texto_str.replace('[', '').replace(']', '').replace('\n', ' ')
    classes_encontradas = re.findall(r"'(.*?)'", texto_str)
    if not classes_encontradas:
        classes_encontradas = [c.strip() for c in texto_str.split('  ') if c.strip()]

    return [c.strip().upper() for c in classes_encontradas if c.strip()]

In [ ]:
model = SentenceTransformer("neuralmind/bert-base-portuguese-cased", device=device)

#reduzindo para 256 tokens para não estourar a memória da GPU T4, mas eu consegui rodar com 512 na versao q entreguei
model.max_seq_length = 256

EPOCAS_TETO = 4
PARES_POR_EPOCA = 12000
train_loss = losses.MultipleNegativesRankingLoss(model=model)

df_train_real = pd.read_parquet("trainC.parquet")
df_labels = pd.read_csv("label_map_enriquecido.csv")

#map as descrições enriquecidas do Ollama
df_labels['label_name_clean'] = df_labels['label_name'].astype(str).str.strip().str.upper()
mapa_textos_ricos = dict(zip(df_labels['label_name_clean'], df_labels['texto_busca']))

print(f"\nTreinamento ({EPOCAS_TETO} épocas)...")
tempo_total_inicio = time.time()

for epoca in range(EPOCAS_TETO):
    print(f"\epoch {epoca + 1}/{EPOCAS_TETO} ---")
    #limpa o cache da GPU
    gc.collect()
    torch.cuda.empty_cache()
    train_examples = []
    df_amostra_epoca = df_train_real.sample(frac=1, random_state=42 + epoca)

    for idx, row in df_amostra_epoca.iterrows():
        if len(train_examples) >= PARES_POR_EPOCA:
            break
        #economizar VRAM
        texto_peticao = str(row.get('texto', ''))[:600]
        coluna_certa = None
        for nome_coluna in row.index:
            if 'label' in str(nome_coluna).lower():
                coluna_certa = nome_coluna
                break

        lista_labels_limpas = limpar_labels_zoados(row[coluna_certa])
        for label_limpo in lista_labels_limpas:
            if label_limpo in mapa_textos_ricos:
                texto_classe_rico = mapa_textos_ricos[label_limpo]
                train_examples.append(InputExample(texts=[texto_peticao, texto_classe_rico]))
                if len(train_examples) >= PARES_POR_EPOCA:
                    break


    #usei 32 no final
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

    #roda o fit para a época atual
    model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=1,
        warmup_steps=100,show_progress_bar=True)
    print(f"epoch {epoca + 1} concluída!")

tempo_final = (time.time() - tempo_total_inicio) / 60
print(f"concluído em {tempo_final:.2f} minutos!")

#salva o modelo final tunado
model.save("pcd")

In [ ]:
model_tunado = SentenceTransformer("pcd", device=device)

df_labels = df_labels.sort_values(by="label_id").reset_index(drop=True)
lista_textos_ricos = df_labels['texto_busca'].astype(str).tolist()
embeddings_labels = model_tunado.encode(lista_textos_ricos,
    show_progress_bar=True,normalize_embeddings=True  #normaliza para usar IndexFlatIP como Similaridade de Cosseno
)

embeddings_labels = np.array(embeddings_labels).astype('float32')
dimensao = embeddings_labels.shape[1]
index = faiss.IndexFlatIP(dimensao)
index.add(embeddings_labels)

#salva o índice em disco
faiss.write_index(index, "pcd_indice_faiss.index")

In [ ]:
index = faiss.read_index("pcd_indice_faiss.index")
lista_nomes_classes = df_labels['label_name'].astype(str).str.strip().str.upper().tolist()
df_val = pd.read_parquet("valC.parquet")

vetores_peticoes = model_tunado.encode(df_val['texto'].astype(str).tolist(),
    show_progress_bar=True,normalize_embeddings=True)
vetores_peticoes = np.array(vetores_peticoes).astype('float32')

K_vizinhos = 8
scores, indices = index.search(vetores_peticoes, K_vizinhos)
y_true = []
for labels_reais in df_val['labels']:
    lista_reais_limpa = limpar_labels_zoados(labels_reais)
    vetor_binario = [1 if classe in lista_reais_limpa else 0 for classe in lista_nomes_classes]
    y_true.append(vetor_binario)
y_true = np.array(y_true)

#melhor threshold baseado no Micro F1
thresholds_para_testar = np.arange(0.25, 0.85, 0.05)
best_micro_f1 = -1
dicionario_metricas_otimo = {}
print(f"{'Thresh':<8} | {'Micro F1':<10} | {'Macro F1':<10} | {'Precision':<10} | {'Recall':<10}")
print("-" * 60)

for t in thresholds_para_testar:
    y_pred = []

    for idx_exemplo in range(len(df_val)):
        vetor_binario_pred = [0] * len(lista_nomes_classes)
        for score, classe_idx in zip(scores[idx_exemplo], indices[idx_exemplo]):
            if score >= t:
                vetor_binario_pred[classe_idx] = 1
        y_pred.append(vetor_binario_pred)
    y_pred = np.array(y_pred)

    mi_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    ma_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    prec = precision_score(y_true, y_pred, average='micro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"{t:.2f}     | {mi_f1:.4f}   | {ma_f1:.4f}   | {prec:.4f}    | {rec:.4f}")

    if mi_f1 >= best_micro_f1:
        best_micro_f1 = mi_f1
        dicionario_metricas_otimo = {
            'threshold': t, 'micro_f1': mi_f1, 'macro_f1': ma_f1, 'precision': prec, 'recall': rec}

if dicionario_metricas_otimo:
    print(f"melhor threshold encontrado: {dicionario_metricas_otimo['threshold']:.2f}")
    print(f"micro F1-Score: {dicionario_metricas_otimo['micro_f1']:.4f}")
    print(f"macro F1-Score: {dicionario_metricas_otimo['macro_f1']:.4f}")

In [ ]:
for i in range(3):
    print(f"\n exemplo {i+1}: {df_val['texto'].iloc[i][:120]}...")
    reais = limpar_labels_zoados(df_val['labels'].iloc[i])
    print(f"real: {reais}")
    sugestoes = [lista_nomes_classes[idx] for idx in indices[i][:3]]
    print(f"top 3: {sugestoes}")
    print(f"scores do Top 3: {scores[i][:3]}")

In [ ]:
#isso aqui foi só pra salvar os arquivos no meu drive
# import shutil
# pasta_destino = "/content/drive/MyDrive/Projeto_CNJ_Validado"
# os.makedirs(pasta_destino, exist_ok=True)

# if os.path.exists("pcd"):
#     if os.path.exists(f"{pasta_destino}/pcd"):
#         shutil.rmtree(f"{pasta_destino}/pcd")
#     shutil.copytree("pcd", f"{pasta_destino}/pcd")

# for arquivo in ["pcd_indice_faiss.index", "label_map_enriquecido.csv"]:
#     if os.path.exists(arquivo):
#         shutil.copy(arquivo, f"{pasta_destino}/")

# if os.path.exists("pcd"):
#     shutil.make_archive("modelo_PRO_entregavel", 'zip', "pcd")
#     files.download("modelo_PRO_entregavel.zip")

# # Baixa o arquivo do FAISS
# if os.path.exists("pcd_indice_faiss.index"):
#     files.download("pcd_indice_faiss.index")
